# Codveda Technology - Data Analytics Internship
**Name:** Divyanshu Gupte

## All 6 Tasks in One Notebook

## 1. Setup

In [ ]:
!pip install pandas numpy matplotlib seaborn scikit-learn nltk textblob wordcloud -q

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns, re
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import *
from textblob import TextBlob
from collections import Counter
import nltk; nltk.download("stopwords",quiet=True); nltk.download("punkt",quiet=True)
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
print("All libraries ready!")

## 2. Upload Dataset Files
Upload: `1) iris.csv`, `4) house Prediction Data Set.csv`, `3) Sentiment dataset.csv`, `churn-bigml-80.csv`

In [ ]:
from google.colab import files
uploaded = files.upload()
print('Uploaded:', list(uploaded.keys()))

---
# L1-T1: Data Cleaning & Preprocessing

In [ ]:
df = pd.read_csv("1) iris.csv")
print("Shape:", df.shape)
print("Missing:", df.isnull().sum().sum())
print("Duplicates:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
df["species"] = df["species"].str.strip().str.lower()
print("After cleaning:", df.shape)
for c in ["sepal_length","sepal_width","petal_length","petal_width"]:
    Q1, Q3 = df[c].quantile(0.25), df[c].quantile(0.75)
    iqr = Q3 - Q1
    lo, hi = Q1 - 1.5*iqr, Q3 + 1.5*iqr
    print(f"{c}: {((df[c]<lo)|(df[c]>hi)).sum()} outliers")
print(df.describe())
df.to_csv("iris_cleaned.csv", index=False)
print("Saved iris_cleaned.csv")

---
# L1-T2: Exploratory Data Analysis

In [ ]:
df = pd.read_csv("iris_cleaned.csv")
nc = ["sepal_length","sepal_width","petal_length","petal_width"]
print(df.describe())
print("Correlation:")
print(df[nc].corr())
df.hist(figsize=(10,8), bins=15, edgecolor="black")
plt.tight_layout(); plt.show()
fig, axes = plt.subplots(2,2,figsize=(10,8))
for ax,col in zip(axes.ravel(), nc):
    sns.boxplot(data=df, x="species", y=col, ax=ax)
plt.tight_layout(); plt.show()
sns.pairplot(df, hue="species", diag_kind="kde")
plt.show()
sns.heatmap(df[nc].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.show()

---
# L2-T1: Regression Analysis

In [ ]:
raw = pd.read_csv("4) house Prediction Data Set.csv", header=None, sep=r"\s+")
cols = ["CRIM","ZN","INDUS","CHAS","NOX","RM","AGE","DIS","RAD","TAX","PTRATIO","B","LSTAT","MEDV"]
raw.columns = cols
X, y = raw.drop("MEDV", axis=1), raw["MEDV"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
m = LinearRegression()
m.fit(X_train, y_train)
p = m.predict(X_test)
print("R2:", r2_score(y_test, p))
print("RMSE:", np.sqrt(mean_squared_error(y_test, p)))
print("MAE:", mean_absolute_error(y_test, p))
for n,c in zip(cols[:-1], m.coef_):
    print(f"{n:8s}: {c:+.4f}")
plt.scatter(y_test, p, alpha=0.6)
plt.plot([y.min(), y.max()], [y.min(), y.max()], "r--", lw=2)
plt.xlabel("Actual"); plt.ylabel("Predicted")
plt.title("Actual vs Predicted")
plt.show()

---
# L2-T3: K-Means Clustering

In [ ]:
df = pd.read_csv("iris_cleaned.csv")
X = df[["sepal_length","sepal_width","petal_length","petal_width"]]
Xs = StandardScaler().fit_transform(X)
inertias = []
for k in range(1, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(Xs)
    inertias.append(km.inertia_)
plt.plot(range(1,11), inertias, "bo-")
plt.xlabel("k"); plt.ylabel("Inertia")
plt.title("Elbow Method")
plt.grid(True, alpha=0.3)
plt.show()
km = KMeans(n_clusters=3, random_state=42, n_init=10)
km.fit(Xs)
df["cluster"] = km.labels_
print("Cross-tabulation:")
print(pd.crosstab(df["species"], df["cluster"]))
pca = PCA(n_components=2)
X_pca = pca.fit_transform(Xs)
plt.scatter(X_pca[:,0], X_pca[:,1], c=km.labels_, cmap="viridis", edgecolor="k", s=60)
centers = pca.transform(km.cluster_centers_)
plt.scatter(centers[:,0], centers[:,1], marker="X", s=200, c="red", edgecolor="k")
plt.title("K-Means Clusters")
plt.show()

---
# L3-T1: Predictive Modeling - Classification

In [ ]:
df = pd.read_csv("churn-bigml-80.csv")
print("Shape:", df.shape)
df["International plan"] = (df["International plan"]=="Yes").astype(int)
df["Voice mail plan"] = (df["Voice mail plan"]=="Yes").astype(int)
df["State"] = LabelEncoder().fit_transform(df["State"])
df["Churn"] = df["Churn"].astype(int)
print("Churn rate:", df["Churn"].mean())
X, y = df.drop("Churn", axis=1), df["Churn"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
sc = StandardScaler()
Xtr = sc.fit_transform(X_train)
Xte = sc.transform(X_test)
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42)
}
for name, model in models.items():
    model.fit(Xtr, y_train)
    pred = model.predict(Xte)
    print(f"\n=== {name} ===")
    print(classification_report(y_test, pred, target_names=["No Churn","Churn"]))
rf = RandomForestClassifier(random_state=42)
rf.fit(Xtr, y_train)
pred = rf.predict(Xte)
print(f"\nAccuracy: {accuracy_score(y_test, pred):.3f}")
print(f"F1: {f1_score(y_test, pred):.3f}")
fi = pd.DataFrame({"feature": X.columns, "importance": rf.feature_importances_})
fi = fi.sort_values("importance", ascending=False)
plt.barh(fi["feature"][:10], fi["importance"][:10], color="steelblue")
plt.gca().invert_yaxis()
plt.xlabel("Importance")
plt.title("Top 10 Features")
plt.tight_layout()
plt.show()

---
# L3-T3: NLP Sentiment Analysis

In [ ]:
df = pd.read_csv("3) Sentiment dataset.csv")
print("Shape:", df.shape)
sw = set(stopwords.words("english"))
stemmer = PorterStemmer()
def clean_text(text):
    text = re.sub(r"http\S+|www\S+|https\S+", "", str(text).lower())
    text = re.sub(r"@\w+|#\w+", "", text)
    text = re.sub(r"[^a-z\s]", "", text)
    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w not in sw and len(w) > 2]
    return " ".join(stemmer.stem(w) for w in tokens)
df["clean"] = df["Text"].apply(clean_text)
def get_sentiment(text):
    p = TextBlob(text).sentiment.polarity
    if p > 0.1: return "positive"
    elif p < -0.1: return "negative"
    else: return "neutral"
df["pred"] = df["Text"].apply(get_sentiment)
print(df["pred"].value_counts())
print()
from wordcloud import WordCloud
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, s in zip(axes, ["positive", "negative", "neutral"]):
    text = " ".join(df[df["pred"]==s]["clean"])
    if text.strip():
        wc = WordCloud(width=400, height=300, max_words=50).generate(text)
        ax.imshow(wc, interpolation="bilinear")
    ax.set_title(s.capitalize())
    ax.axis("off")
plt.tight_layout()
plt.show()
for s in ["positive", "negative", "neutral"]:
    words = " ".join(df[df["pred"]==s]["clean"]).split()
    print(f"\n{s.capitalize()}: {Counter(words).most_common(10)}")

---
# All 6 Tasks Complete!

Thank you **Codveda Technology**!